# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Thallysson100/FlyRank_AI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook establishes the formal data contract for predicting content performance decline across client domains.

## 1. Unit of analysis + time window

*One row = one content item (`content_hash_id`) for a client (`client_hash_id`).*

One row represents one pseudonymized content item (`content_hash_id`) for a specific client (`client_hash_id`). The time window uses a trailing 90-day feature window ($T - 120$ days to $T - 30$ days) to predict outcomes in the subsequent 30-day label window ($T - 30$ days to $T$), where $T$ is the maximum recorded `report_date` in the daily performance fact table.

In [4]:
import duckdb
import os, getpass

# Setup connection and authenticate
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token: ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

# Verify the grain: unique content_hash_id per row in dimension table
grain_check = con.sql(f"""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT content_hash_id) AS unique_content_ids
    FROM {TABLES['dim_content']}
""").df()

print("Unit of Analysis Verification:")
print(grain_check)


Unit of Analysis Verification:
   total_rows  unique_content_ids
0      519606              519606


## 2. Fields: feature / label / context / excluded

*   **Feature**: 
    *   `imp_prev90`: Sum of GSC impressions during the 90-day feature window ($T - 120$ to $T - 30$).
    *   `clk_prev90`: Sum of GSC clicks during the 90-day feature window.
    *   `visible_queries`: Count of unique queries ranking for the page.
    *   `impressions_prev30_total`: Safe sum of impressions from the query table preceding the label window.
*   **Label**: 
    *   `is_declining_label`: Binary label set to `1` when impressions in the subsequent 30-day window drop by $>20\%$ compared to the average monthly rate in the 90-day feature window ($[imp\_last30 < 0.8 \times (imp\_prev90 / 3)]$).
*   **Context**: 
    *   `content_hash_id`: Unique content page key.
    *   `client_hash_id`: Unique client key used strictly for `GroupShuffleSplit` during model validation.
*   **Excluded**: 
    *   `trend_direction` / `trend_pct`: Direct sources of label computation; inclusion causes target leakage.
    *   `provider_used` / `model_used`: Metadata properties designated as non-feature descriptors.
    *   `impressions_90d`, `last_30d`, `rare_share`, and `anon_share` inside `fact_content_query_90d`: Overlaps temporally with the target label evaluation period causing leakage.

In [21]:
# Verify schema and feature presence with SAFE non-leaking query features and correct grain
schema_verify = con.sql(f"""
    WITH safe_query_features AS (
    SELECT 
        content_hash_id,
        COUNT(DISTINCT query_hash_id) AS query_count_prev30,
        MAX(content_visible_query_count) AS visible_queries,
        SUM(impressions_prev30) AS impressions_prev30_total
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
)
SELECT 
    c.content_hash_id,
    c.client_hash_id,
    q.visible_queries,
    q.impressions_prev30_total
FROM {TABLES['dim_content']} c
LEFT JOIN safe_query_features q 
    ON c.content_hash_id = q.content_hash_id
LIMIT 5;
""").df()

print("Sample Feature & Context Alignment:")
print(schema_verify)

Sample Feature & Context Alignment:
            content_hash_id           client_hash_id  visible_queries  \
0  content_7bde651b9cd42ceb  client_08a6a72ff48e62c0               14   
1  content_7c203d3b70796129  client_08a6a72ff48e62c0                2   
2  content_7c6dc6096625be6e  client_08a6a72ff48e62c0               26   
3  content_7d1847c77fc26961  client_08a6a72ff48e62c0                3   
4  content_7d2f3dbe184f4f31  client_08a6a72ff48e62c0                4   

   impressions_prev30_total  
0                     379.0  
1                       7.0  
2                    2368.0  
3                      40.0  
4                      28.0  


## 3. Verify it with queries (grain, counts, missing values, windows)

Queries verifying grain consistency, observation windows, missingness patterns, and target distribution.

In [22]:
# 1. Time Window Boundaries
window_verify = con.sql(f"""
    SELECT 
        MAX(report_date) AS max_date,
        MAX(report_date) - INTERVAL 30 DAY AS label_window_start,
        MAX(report_date) - INTERVAL 120 DAY AS feature_window_start
    FROM {TABLES['fact_daily_sample']}
""").df()

print("Time Window Ranges:")
print(window_verify)

# 2. Missingness Audit Across Content Types
missingness_audit = con.sql(f"""
    SELECT 
        c.content_type,
        COUNT(*) AS total_content_items,
        COUNT(q.content_visible_query_count) AS items_with_query_data,
        ROUND(AVG(CASE WHEN q.content_visible_query_count IS NULL THEN 1.0 ELSE 0.0 END) * 100, 2) AS missing_query_pct
    FROM {TABLES['dim_content']} c
    LEFT JOIN {TABLES['fact_query_90d']} q ON c.content_hash_id = q.content_hash_id
    GROUP BY c.content_type
""").df()

print("\nMissingness Audit by Content Type:")
print(missingness_audit)

# 3. Target Distribution (Base Rate Calculation)
target_distribution = con.sql(f"""
    WITH bounds AS (
    SELECT MAX(report_date) AS max_d FROM {TABLES['fact_daily_sample']}
),
aggregates AS (
    SELECT 
        f.content_hash_id,
        SUM(CASE WHEN f.report_date > b.max_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
        SUM(CASE WHEN f.report_date <= b.max_d - INTERVAL 30 DAY 
                 AND f.report_date > b.max_d - INTERVAL 120 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev90
    FROM {TABLES['fact_daily_sample']} f
    CROSS JOIN bounds b
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id
    HAVING imp_prev90 >= 300
)
SELECT 
    COUNT(*) AS total_eligible_items,
    SUM(CASE WHEN imp_last30 < 0.8 * (imp_prev90 / 3.0) THEN 1 ELSE 0 END) AS declining_items,
    ROUND(AVG(CASE WHEN imp_last30 < 0.8 * (imp_prev90 / 3.0) THEN 1.0 ELSE 0.0 END), 3) AS base_decline_rate
FROM aggregates;
""").df()

print("\nTarget Base Rate:")
print(target_distribution)

Time Window Ranges:
    max_date label_window_start feature_window_start
0 2026-06-30         2026-05-31           2026-03-02

Missingness Audit by Content Type:
         content_type  total_content_items  items_with_query_data  \
0  comparison article                 5568                   3346   
1      feedly article                63476                   7753   
2     keyword article              2730958                2403149   

   missing_query_pct  
0              39.91  
1              87.79  
2              12.00  

Target Base Rate:
   total_eligible_items  declining_items  base_decline_rate
0                     0              NaN                NaN


> **Note on Sample Data Limits:**
> The `fact_daily_sample` table used for local testing is a 30-day snapshot slice ending at $T$ (`2026-06-30`). Because evaluating the target requires daily performance records spanning the 90-day feature window from $T - 120$ days to $T - 30$ days (`2026-03-02` to `2026-05-31`), `imp_prev90` sums to `0` for all rows in this sample slice. Consequently, `total_eligible_items` evaluates to `0` under the `HAVING imp_prev90 >= 300` volume threshold. This target distribution logic is designed to execute against the full, unpartitioned `fact_content_daily_performance` table in production, where complete multi-month historical coverage exists.
> 
>

## 4. Data limits

*   **Unbalanced Panel**: Per-client tracking history varies significantly (`gsc_data_start` ranges across different calendar months). Time-series models must evaluate windowing on per-client relative dates rather than global fixed calendar cutoffs.
*   **Three-Valued GA4 Access Flags**: Early history lacks GA4 integration. GA4 flags are three-valued (`TRUE`, `FALSE`, or `NULL`). Queries filtering on missing engagement metrics must use `IS TRUE` / `IS NOT TRUE` rather than binary equals syntax.
*   **Query Table Temporal Overlap**: `fact_content_query_90d` represents a static 90-day aggregate ending at snapshot export time. Raw impression totals inside this table cover the label period and cannot be directly included as unadjusted features.

In [15]:
# 1. Inspect Client Panel Unbalance
client_panel = con.sql(f"""
    SELECT 
        MIN(gsc_data_start) AS earliest_gsc,
        MAX(gsc_data_start) AS latest_gsc,
        COUNT(DISTINCT client_hash_id) AS total_clients
    FROM {TABLES['dim_clients']}
""").df()

print("Client History Breadth:")
print(client_panel)

# 2. Check GA4 3-Valued Access Flag Distributions
ga4_flag_distribution = con.sql(f"""
    SELECT 
        access_profile,
        COUNT(*) AS client_count,
        SUM(CASE WHEN ga4_data_start IS NULL THEN 1 ELSE 0 END) AS null_ga4_starts
    FROM {TABLES['dim_clients']}
    GROUP BY access_profile
""").df()

print("\nGA4 Access Profile Breakdown:")
print(ga4_flag_distribution)


Client History Breadth:
  earliest_gsc latest_gsc  total_clients
0   2025-01-27 2026-06-02            104

GA4 Access Profile Breakdown:
                         access_profile  client_count  null_ga4_starts
0                              ga4_only             1              1.0
1                              gsc_only            14             14.0
2                           gsc_and_ga4            53              3.0
3         no_search_or_analytics_access            26             26.0
4  source_only_missing_client_dimension            10              9.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.